# 🗂️ 개인실습 — 로그 파이프라인 적재 (7~8교시, 실제 PostgreSQL에 내 손으로 담고 조회하기)

2~6교시(데모 노트북 또는 강의)에서 관계형 모델·JOIN·인덱스·옵티마이저의 원리를 확인했다면, 이번에는 **이전 시간(로그 분석 실습)의 결과를 진짜 PostgreSQL에 안전하게 담고, SQL로 조회**합니다.

## 📋 이 실습에서 채울 것 — 체크포인트 4개 (CP1~CP4)
- 순서: **스키마 설계(CP1) → 트랜잭션 적재(CP2) → 에러율 조회(CP3) → JOIN 조회(CP4)**

## ⏱️ 예상 시간표

| 단계 | 실습 |
| --- | --- |
| 준비 | course_db 생성 + 연결 (실행만) |
| 실습 1 (CP1) | 스키마 설계 — hourly_error_stats·latency_stats·spike_windows | 
| 실습 2 (CP2) | 트랜잭션으로 적재 — ON CONFLICT·commit |
| 실습 3 (CP3) | 에러율 상위 5개 조회 — NULLIF·ORDER BY·LIMIT |
| 실습 4 (CP4) | 급증 구간 + 응답시간 JOIN |

## ⚠️ 꼭 기억하세요
- 1교시에서 띄운 **`db-pg` 컨테이너**가 켜져 있어야 합니다.
- **비밀번호는 `.env`에서 읽거나, 없으면 기본값 `postgres`를 씁니다** — 원본 스크립트(`load.py`·`queries.py`·`verify.py`)와 동일한 방식입니다.
- 막히면 `lab_dayA/lab/load/troubleshooting.md`를 참고하세요. **정답은 별도 파일** `PostgreSQL_3_로그파이프라인_개인실습_정답.ipynb`에 있습니다.
- 이 노트북을 다 채운 뒤 터미널에서 `python lab/load/verify.py`(인자 없이)를 실행하면 CP1~CP4가 실제로 통과하는지 원본 검증 스크립트로도 확인할 수 있습니다 — 노트북과 같은 `course_db`를 보므로 결과가 일치합니다.


---
## ⚙️ 준비 — course_db 만들기 (실행만 하세요)

`db-pg` 컨테이너 안에 `course_db`가 없으면 만듭니다(이미 2~6교시 데모 노트북을 실행했다면 이미 있습니다 — 재실행 안전). 조회 결과를 한 줄씩 출력하는 `run()`도 준비합니다.


In [ ]:
# db-pg 컨테이너 안 PostgreSQL에 'course_db'를 만듭니다 (없을 때만 생성 → 재실행 안전)
# 아래는 Windows(cmd.exe) 기준입니다 — Linux/macOS는 바로 아래 주석 처리된 줄을 대신 쓰세요.
!docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='course_db'" | findstr /C:"1" >NUL || docker exec db-pg psql -U postgres -c "CREATE DATABASE course_db"

# ---- Linux/macOS ----
# !docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='course_db'" | grep -q 1 || docker exec db-pg psql -U postgres -c "CREATE DATABASE course_db"


In [ ]:
# 조회 결과를 한 줄씩 출력하는 도우미 (원본 스크립트에는 없지만, 노트북에서 결과를 바로 보기 위해 추가했습니다)
def run(sql, params=None):
    cur.execute(sql, params)
    rows = cur.fetchall()
    for row in rows:
        print(row)
    return rows

print("준비 완료 — course_db 확인/생성됨.")


---
## 🧪 실습 1 (CP1) — 스키마 설계: hourly_error_stats·latency_stats·spike_windows ⭐

**시나리오**: 이전 시간(로그 분석) 결과는 세 종류입니다 — 시간대별 에러 건수, 응답시간 백분위(p50/p95/p99), 에러 급증 구간. 이 셋을 각각 담을 표를 만듭니다.

**요구사항**: 아래 코드의 빈칸(`_____`)을 채워, PK·CHECK·UNIQUE 제약을 포함한 세 표를 만드세요. (원본: `lab/load/skeleton/schema.sql`)

**기대 출력**: `information_schema.tables` 조회에 `hourly_error_stats`·`latency_stats`·`spike_windows` 3행이 알파벳 순으로 나오면 성공.


In [ ]:
# 🧪 실습 1 (CP1) — 빈칸(_____)을 채우세요. 원본: lab/load/skeleton/schema.sql
import os
import psycopg
from dotenv import load_dotenv

load_dotenv()
conn = psycopg.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    dbname=os.getenv("POSTGRES_DB", "course_db"),
    user=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD", "postgres"),
)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS spike_windows")
cur.execute("DROP TABLE IF EXISTS latency_stats")
cur.execute("DROP TABLE IF EXISTS hourly_error_stats")

# ① hourly_error_stats : 시간대별 에러 집계
cur.execute("""
    CREATE TABLE hourly_error_stats (
        id          _____       PRIMARY KEY,          -- 자동 증가 PK
        log_date    DATE        NOT NULL,
        hour        _____       NOT NULL              -- 0~23 사이 정수
                        CHECK (hour BETWEEN _____ AND _____),
        error_count INTEGER     NOT NULL DEFAULT 0,
        total_count INTEGER     NOT NULL DEFAULT 0,
        UNIQUE (log_date, _____)                      -- 날짜+시간 조합 중복 불가
    )
""")

# ② latency_stats : 시간대별 응답시간 백분위
cur.execute("""
    CREATE TABLE latency_stats (
        id       SERIAL        PRIMARY KEY,
        log_date DATE          NOT NULL,
        hour     SMALLINT      NOT NULL
                     CHECK (hour BETWEEN 0 AND 23),
        p50      _____,                               -- NULL 허용 (ms 단위)
        p95      NUMERIC(10,2),
        p99      NUMERIC(10,2),
        UNIQUE (log_date, hour)
    )
""")

# ③ spike_windows : 에러 급증 구간
cur.execute("""
    CREATE TABLE _____ (                              -- 테이블 이름을 채우세요
        id          SERIAL       PRIMARY KEY,
        log_date    DATE         NOT NULL,
        start_hour  SMALLINT     NOT NULL,
        end_hour    SMALLINT     NOT NULL,
        peak_count  INTEGER      NOT NULL,
        spike_ratio NUMERIC(5,4)
    )
""")
conn.commit()

run("""
    SELECT table_name
    FROM   information_schema.tables
    WHERE  table_schema = 'public'
      AND  table_name IN ('hourly_error_stats', 'latency_stats', 'spike_windows')
    ORDER BY table_name
""")


### 💡 힌트 (실습 1)
- **자동 증가 PK**: 3교시·4교시 데모에서 이미 썼던, INSERT 때 값을 안 줘도 1,2,3...으로 채워지는 타입입니다.
- **hour 타입**: 0~23이면 충분히 작은 정수입니다 — `INTEGER`보다 공간을 덜 쓰는 정수 타입을 씁니다.
- **CHECK 범위**: 교안 3교시의 그 조건 그대로 — `hour`가 0 이상 23 이하.
- **UNIQUE 대상**: "날짜+시간 조합이 중복되면 안 된다"는 뜻입니다 — 이미 `log_date`는 UNIQUE에 들어가 있으니 나머지 컬럼 하나만 채우면 됩니다.
- **p50 타입**: `p95`·`p99`와 같은 타입입니다(응답시간, ms 단위, 소수 둘째 자리까지).
- **세 번째 표 이름**: 이 절의 제목과 CP1 검증 쿼리의 `table_name IN (...)` 목록에 이미 힌트가 있습니다.


---
## 🧪 실습 2 (CP2) — 트랜잭션으로 안전하게 적재하기 ⭐

**시나리오**: 이전 시간 결과 파일(`lab/load/data/result_sample.json`)을 방금 만든 세 표에 적재합니다. `cur = conn.cursor()`가 열린 순간부터 이미 트랜잭션 안입니다 — 셋 중 하나라도 실패하면 `conn.commit()`에 도달하지 못해 전부 롤백됩니다.

**요구사항**: 아래 코드의 빈칸을 채우세요. (원본: `lab/load/skeleton/load.py`)
- `result["_____"]` → 적재 대상 날짜가 담긴 키
- `ON CONFLICT (log_date, hour) DO _____` → 중복이면 무시
- `lats.get("_____")` → p50 값을 꺼내는 키
- `spike["_____"]` → start_hour 값을 꺼내는 키
- `conn._____()` → 세 종류 INSERT가 전부 성공했을 때만 영구 저장

**기대 출력**: `hourly_error_stats` 24행, `latency_stats` 24행, `spike_windows` 4행.


In [ ]:
# 🧪 실습 2 (CP2) — 빈칸(_____)을 채우세요. 원본: lab/load/skeleton/load.py
import json

# 노트북은 lab_dayA/notebooks/에 있다는 전제 — 원본 스크립트 기준 상대경로(lab/load/data/...)에 ../를 붙였습니다.
result_file = "../lab/load/data/result_sample.json"
with open(result_file, encoding="utf-8") as f:
    result = json.load(f)

log_date = result["_____"]   # "log_date" 키를 채우세요
print(f"적재 대상 날짜: {log_date}")

# ① hourly_error_stats 적재
for hour_str, stats in result["hourly_errors"].items():
    cur.execute("""
        INSERT INTO hourly_error_stats (log_date, hour, error_count, total_count)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (log_date, hour) DO _____   -- 중복이면 무시
    """, (log_date, int(hour_str), stats["error_count"], stats["total_count"]))

# ② latency_stats 적재
for hour_str, lats in result.get("latency", {}).items():
    cur.execute("""
        INSERT INTO latency_stats (log_date, hour, p50, p95, p99)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (log_date, hour) DO NOTHING
    """, (log_date, int(hour_str),
          lats.get("_____"),   # p50
          lats.get("p95"),
          lats.get("p99")))

# ③ spike_windows 적재
for spike in result.get("spike_windows", []):
    cur.execute("""
        INSERT INTO spike_windows (log_date, start_hour, end_hour, peak_count, spike_ratio)
        VALUES (%s, %s, %s, %s, %s)
    """, (log_date,
          spike["_____"],   # start_hour
          spike["end_hour"],
          spike["peak_count"],
          spike.get("spike_ratio")))

conn._____()   # commit 또는 rollback

run("""
    SELECT 'hourly_error_stats' AS 테이블, COUNT(*) AS 행수 FROM hourly_error_stats
    UNION ALL
    SELECT 'latency_stats',  COUNT(*) FROM latency_stats
    UNION ALL
    SELECT 'spike_windows',  COUNT(*) FROM spike_windows
""")


### 💡 힌트 (실습 2)
- `result_sample.json`을 열어보면 최상위에 `"log_date": "2016-11-09"`가 보입니다 — 그 키 이름을 그대로 쓰세요.
- "이미 있으면 무시"에 해당하는 `ON CONFLICT ... DO` 뒤에 오는 SQL 키워드입니다 — 영어 단어 그대로(대문자 7글자)입니다.
- `lats`는 `{"p50": ..., "p95": ..., "p99": ...}` 형태의 딕셔너리입니다. 바로 아래 줄의 `lats.get("p95")`와 같은 방식으로 `p50`을 꺼내세요.
- `spike`는 `{"start_hour": ..., "end_hour": ..., ...}` 형태입니다. 바로 아래 줄의 `spike["end_hour"]`와 같은 방식입니다.
- ⭐ **가장 중요한 빈칸**: 세 종류 INSERT가 모두 성공했을 때 트랜잭션을 영구 저장하는 메서드입니다 — 이 실습 제목에 이미 답이 있습니다.


---
## 🧪 실습 3 (CP3) — 시간대별 에러율 상위 5개

**시나리오**: 이전 시간 파이썬으로 계산했던 "에러율 상위 시간대"를 SQL 한 줄로 재현합니다.

**요구사항**: 빈칸을 채우세요. (원본: `lab/load/skeleton/queries.py`)
- `NULLIF(total_count, _____)` → 0으로 나누기 방지
- `ORDER BY error_rate_pct _____` → 내림차순
- `LIMIT _____` → 상위 5개

**기대 출력**: 03시가 약 12.77%로 1위, 나머지는 훨씬 낮은 값.


In [ ]:
# 🧪 실습 3 (CP3) — 빈칸(_____)을 채우세요. 원본: lab/load/skeleton/queries.py
LOG_DATE = "2016-11-09"

run("""
    SELECT
        hour,
        error_count,
        total_count,
        ROUND(error_count::NUMERIC / NULLIF(total_count, _____) * 100, 2) AS error_rate_pct
    FROM   hourly_error_stats
    WHERE  log_date = %s
    ORDER BY error_rate_pct _____    -- DESC (내림차순)
    LIMIT _____                      -- 상위 5개
""", (LOG_DATE,))


### 💡 힌트 (실습 3)
- "0으로 나누면 안 된다"는 `NULLIF`의 두 번째 인자에 어떤 숫자를 넣어야 할지로 표현됩니다.
- "상위"는 큰 값이 먼저 나와야 한다는 뜻입니다.
- "5개"는 결과 행 수를 제한하는 키워드 하나로 해결됩니다.


---
## 🧪 실습 4 (CP4) — 에러 급증 구간 + 응답시간 JOIN

**시나리오**: 급증 구간(`spike_windows`)과 그 시간대의 응답시간(`latency_stats`)을 함께 봅니다.

**요구사항**: 빈칸을 채우세요. (원본: `lab/load/skeleton/queries.py`)
- `_____  latency_stats l` → JOIN 종류(양쪽 표에 모두 있는 시간대만 필요합니다)
- `l.hour BETWEEN s.start_hour AND s._____` → end_hour

**기대 출력**: 4개 급증 구간(3·14·19·22시) 모두 응답시간과 함께 나옵니다.


In [ ]:
# 🧪 실습 4 (CP4) — 빈칸(_____)을 채우세요. 원본: lab/load/skeleton/queries.py
run("""
    SELECT
        s.start_hour,
        s.end_hour,
        s.peak_count,
        ROUND(s.spike_ratio * 100, 2) AS spike_rate_pct,
        l.hour,
        l.p50,
        l.p95,
        l.p99
    FROM   spike_windows s
    _____  latency_stats l               -- JOIN 종류를 채우세요
        ON s.log_date = l.log_date
       AND l.hour BETWEEN s.start_hour AND s._____   -- end_hour
    WHERE  s.log_date = %s
    ORDER BY s.start_hour, l.hour
""", (LOG_DATE,))


### 💡 힌트 (실습 4)
- 4교시 데모에서 다룬 두 JOIN 중 "양쪽 모두에 있는 행만" 남기는 쪽입니다. `INNER`는 생략해도 됩니다(기본값).
- `BETWEEN`은 "시작 ~ 끝"을 나타냅니다. 이미 왼쪽에 `s.start_hour`가 있으니, 오른쪽엔 짝이 되는 컬럼을 넣습니다.


---
## 정리 및 자가체크

- [ ] CP1: `schema.sql`의 빈칸을 채우고 세 표가 모두 생성됐다
- [ ] CP2: `load.py`의 빈칸을 채우고 24/24/4행이 적재됐다
- [ ] CP3: 03시 에러율이 약 12.77%로 1위인 것을 확인했다
- [ ] CP4: 급증 구간 4개가 응답시간과 함께 JOIN되는 것을 확인했다
- [ ] (선택) 터미널에서 `python lab/load/verify.py`를 실행해 원본 검증 스크립트로도 통과를 확인했다

### 회고
- 오늘 가장 헷갈렸던 빈칸은 무엇이었나요?
- `conn.commit()`이 왜 트랜잭션의 끝인지 옆 사람에게 설명할 수 있나요?

막혔던 부분은 `PostgreSQL_3_로그파이프라인_개인실습_정답.ipynb`에서 확인하세요.
